# NB04 — Retrieval Artefact Construction (Facial Skincare)

**Purpose.** Build the query-side retrieval artefacts: leakage-guarded item documents (dense/sparse), item facets and flags, the Global Review item graph (long/edges/nodes), and temporal precomputes.
**Inputs.** `data/processed/items/face_item_schema{,_full}.parquet` (NB02), `data/raw/reviews_…parquet`.
**Outputs.** `data/processed/items/` — `face_item_docs{,_dense,_sparse}.parquet`, `face_items_facets.parquet`, graph artefacts (`face_item_graph_long/edges`, product/entity nodes), temporal artefacts; contract export + manifest.
**Position.** NB02 → **this** → NB07/NB08 (retrieval), NB09 (candidate pools).
**Run notes.** Leakage guards precede document construction; contract-check cells are QC, not configuration. Executed record.


# 04. Retrieval artifact construction - Facial Skincare


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Imports ====
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

In [3]:
# ==== Config ====
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data" / "processed" / "items"
TEMPORAL_ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed" / "temporal"
RAW_REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Skin_Care_Face_W2_2019_2023.parquet"

SCHEMA_PATH = PROCESSED_ITEMS_DIR / "face_item_schema.parquet"
ACTIVE_SCHEMA_PATH = PROCESSED_ITEMS_DIR / "face_item_schema_full.parquet"

ITEM_DOCS_DENSE_PATH = PROCESSED_ITEMS_DIR / "face_item_docs_dense.parquet"
ITEM_DOCS_SPARSE_PATH = PROCESSED_ITEMS_DIR / "face_item_docs_sparse.parquet"
ITEM_DOCS_PATH = PROCESSED_ITEMS_DIR / "face_item_docs.parquet"
ITEMS_FACETS_PATH = PROCESSED_ITEMS_DIR / "face_items_facets.parquet"
ITEM_GRAPH_LONG_PATH = PROCESSED_ITEMS_DIR / "face_item_graph_long.parquet"
ITEM_KG_EDGES_PATH = PROCESSED_ITEMS_DIR / "face_item_kg_edges.parquet"
GRAPH_EDGES_PATH = PROCESSED_ITEMS_DIR / "face_item_graph_edges.parquet"
PRODUCT_NODES_PATH = PROCESSED_ITEMS_DIR / "face_item_graph_product_nodes.parquet"
ENTITY_NODES_PATH = PROCESSED_ITEMS_DIR / "face_item_graph_entity_nodes.parquet"
FACET_VOCAB_PATH = PROCESSED_ITEMS_DIR / "face_facet_vocab.json"
FACET_VOCAB_TABLE_PATH = PROCESSED_ITEMS_DIR / "face_facet_vocab.parquet"
FACET_COVERAGE_PATH = PROCESSED_ITEMS_DIR / "face_items_facets_coverage.csv"
DOC_COVERAGE_PATH = PROCESSED_ITEMS_DIR / "face_item_docs_coverage.csv"
CORE_FACET_FILTER_AUDIT_PATH = PROCESSED_ITEMS_DIR / "face_core_facet_filter_audit.csv"
MANIFEST_PATH = PROCESSED_ITEMS_DIR / "retrieval_artifact_manifest_face.json"

ITEM_REVIEW_TIMESTAMP_INDEX_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_review_timestamp_index.parquet"
ITEM_DAILY_REVIEW_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_daily_review_counts.parquet"
ENTITY_DAILY_REVIEW_COUNTS_PATH = TEMPORAL_ARTIFACT_DIR / "face_entity_daily_review_counts.parquet"
ITEM_TEMPORAL_SUMMARY_PATH = TEMPORAL_ARTIFACT_DIR / "face_item_temporal_summary.parquet"
TEMPORAL_MANIFEST_PATH = TEMPORAL_ARTIFACT_DIR / "temporal_artifact_manifest_face.json"

COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH = PROCESSED_ITEMS_DIR / "face_retrieval_artifact_common_contract.json"
COMMON_FACET_ROLE_COVERAGE_PATH = PROCESSED_ITEMS_DIR / "face_facet_role_coverage.csv"
COMMON_FRAMEWORK_FLAG_SELF_CHECK_PATH = PROCESSED_ITEMS_DIR / "common_framework_flag_self_check_face.csv"

TEMPORAL_LEAKAGE_RULE = "review_timestamp_ms < target_timestamp_ms"

PRODUCTION_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PRODUCTION_FACET_EXPORT_OPTION = "B_all_rows_with_reliable_flags"
CORE_GRAPH_MASK_DESCRIPTION = (
    "is_product_functional_facet == True and is_query_safe == True and "
    "is_brand == False and is_review_derived == False and "
    "is_generic_category_anchor == False and is_generic_utility_token == False and "
    "is_context_dependent_utility_token == False and is_metadata_facet_source == True and "
    "is_disallowed_nonfacet_source == False"
)
BRAND_GRAPH_MASK_DESCRIPTION = (
    "facet_role == 'brand' and is_brand == True and is_review_derived == False and "
    "is_generic_category_anchor == False and is_generic_utility_token == False and "
    "is_context_dependent_utility_token == False and "
    "is_metadata_facet_source == True and is_disallowed_nonfacet_source == False"
)
GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION = (
    "is_core_graph_facet == True OR is_brand_graph_facet == True OR "
    "(is_review_derived == True and is_brand == False and "
    "is_generic_category_anchor == False and is_generic_utility_token == False and "
    "is_context_dependent_utility_token == False and is_disallowed_nonfacet_source == False)"
)


FORBIDDEN_FACET_SOURCE_TERMS = ("identifier", "count", "policy", "diagnostic")

CATEGORY_REQUIRED_FACET_COLS = [
    "facet_category_text",
    "facet_form_text",
    "facet_skin_type_text",
    "facet_ingredient_text",
    "facet_benefit_text",
    "facet_claim_text",
    "facet_scent_text",
]
CATEGORY_OPTIONAL_FACET_SPECS = [
    ("facet_skin_type_text", "skin_type", "HAS_SKIN_TYPE"),
    ("facet_scent_text", "scent", "HAS_SCENT"),
]

EXPECTED_FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
EXPECTED_BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"
EXPECTED_IDENTIFIER_POLICY = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

PROCESSED_ITEMS_DIR.mkdir(parents=True, exist_ok=True)
TEMPORAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", SCHEMA_PATH)
print("Input:", RAW_REVIEWS_PATH)
print("Output:", ITEM_DOCS_PATH)


Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_schema.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_docs.parquet


### Global Review Retrieval Artifact Framework

Production item documents use the merged Global Review representation: catalog
metadata, the separate brand facet, product-functional facets, and normalized item-level signals extracted
from reviews before the frozen training cutoff. The item-facet parquet retains all
audited rows with reliable flags (Option B), while the production graph combines
the validated metadata functional-facet mask, a separate brand facet channel, and
historical review-derived facet rows. Brand remains excluded from synthetic-query
evidence but is retained in catalog retrieval, graph, and user-profile representations.
Generic terms, raw review text, and user-specific history remain excluded from the
query-only evidence path.


In [4]:
# ==== Common Retrieval Artifact Contract ====
COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION = "common_retrieval_artifact_framework_v5_global_review_brand_profile"

COMMON_FACET_TYPE_TO_ROLE = {
    "brand": "brand",
    "category": "category_or_product_type",
    "product_type": "category_or_product_type",
    "form": "form_texture",
    "formulation": "ingredient_or_composition",
    "texture": "form_texture",
    "skin_type": "target_context",
    "usage_target": "target_context",
    "scent": "sensory",
    "ingredient": "ingredient_or_composition",
    "benefit": "need_benefit_concern",
    "concern": "need_benefit_concern",
    "claim": "claim_constraint",
    "review_reputation_concern": "review_derived_signal",
    "review_reputation_skin_type": "review_derived_signal",
    "review_reputation_benefit": "review_derived_signal",
    "review_reputation_ingredient": "review_derived_signal",
    "review_reputation_product_form_texture": "review_derived_signal",
    "review_reputation_product_type": "review_derived_signal",
    "review_reputation_form": "review_derived_signal",
    "review_reputation_formulation": "review_derived_signal",
    "review_reputation_texture": "review_derived_signal",
    "review_reputation_usage_target": "review_derived_signal",
    "review_reputation_claim_diet": "review_derived_signal",
    "review_reputation_flavor": "review_derived_signal",
    "review_reputation": "review_derived_signal",
}
COMMON_PRODUCT_FUNCTIONAL_ROLES = {
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
}
COMMON_GENERIC_CATEGORY_ANCHORS = {
    "skincare", "face", "facial", "skin care", "skincare routine", "skin care routine"
}
COMMON_GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "formula", "blend", "complex", "product", "solution",
}
COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS = {
    "daily", "natural", "wellness", "health", "care", "routine",
}
COMMON_SPECIFIC_PHRASE_EXCEPTIONS = {
    "morning routine", "night routine", "skin barrier support",
    "hydration support", "sensitive skin care", "acne care",
}

COMMON_RETRIEVAL_ARTIFACT_CONTRACT = {
    "category_id": "face",
    "framework_version": COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION,
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "dense_source": "dense_text",
    "sparse_source": "sparse_text",
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "production_core_facet_mask": CORE_GRAPH_MASK_DESCRIPTION,
    "production_brand_facet_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "production_global_review_facet_mask": GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION,
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_in_functional_graph": False,
    "brand_graph_enabled": True,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
    "raw_review_text_exported": False,
    "facet_type_to_role": COMMON_FACET_TYPE_TO_ROLE,
    "generic_category_anchors": sorted(COMMON_GENERIC_CATEGORY_ANCHORS),
    "generic_utility_tokens": sorted(COMMON_GENERIC_UTILITY_TOKENS),
    "context_dependent_utility_tokens": sorted(COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS),
    "specific_phrase_exceptions": sorted(COMMON_SPECIFIC_PHRASE_EXCEPTIONS),
    "product_functional_roles": sorted(COMMON_PRODUCT_FUNCTIONAL_ROLES),
    "query_audit_note": (
        "Routine is context-dependent in Facial queries: morning/night routine are specific target-context "
        "phrases, while skincare routine remains a generic category phrase."
    ),
}


In [5]:
# ==== Leakage Guards ====
def normalize_space(value):
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = str(value).replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_entity_value(value):
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9가-힣\s_\-+/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_values(value):
    text = normalize_space(value)
    if not text:
        return []

    values = []
    for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text):
        part = normalize_space(part)
        if part:
            values.append(part)

    out = []
    seen = set()
    for value in values:
        key = normalize_entity_value(value)
        if key and key not in seen:
            seen.add(key)
            out.append(value)
    return out


def make_entity_node_id(facet_type, value):
    return f"entity::{facet_type}::{normalize_entity_value(value)}"


def non_empty_ratio(series):
    if len(series) == 0:
        return 0.0
    return float(series.fillna("").astype(str).str.strip().ne("").mean())


def assert_no_leakage_columns(df, frame_name):
    exact_forbidden = {
        "target_review_text",
        "heldout_review_text",
        "review_text",
        "review_body",
        "raw_review_text",
        "rating",
        "average_rating",
        "rating_number",
        "rating_count",
        "rating_num",
        "sentiment",
        "helpful_vote",
        "helpful_votes",
        "total_vote",
        "prompt",
        "response",
        "llm_response",
        "query_evidence",
    }
    forbidden_prefixes = (
        "target_review_",
        "heldout_review_",
        "raw_review_",
        "review_text_",
        "review_body_",
        "review_evidence_",
    )
    blocked = [
        str(col)
        for col in df.columns
        if str(col) in exact_forbidden
        or any(str(col).startswith(prefix) for prefix in forbidden_prefixes)
    ]
    if blocked:
        raise RuntimeError(f"{frame_name} contains forbidden leakage columns: {blocked}")


def assert_allowed_output_path(path):
    path_str = str(path).lower()
    blocked_terms = [
        "target_review_text",
        "heldout_review_text",
        "raw_review",
        "review_body",
        "full_history_user",
        "user_profile",
        "user_anchor",
    ]
    if any(term in path_str for term in blocked_terms):
        raise RuntimeError(f"Unsafe output path: {path}")

In [6]:
# ==== Item Schema Loading ====
schema_df = pd.read_parquet(SCHEMA_PATH).copy()

required_schema_columns = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "canonical_metadata_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_dedup_seed",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "facet_policy_version",
    "brand_policy",
    "identifier_policy",
    *CATEGORY_REQUIRED_FACET_COLS,
]
missing_schema_columns = [
    column for column in required_schema_columns
    if column not in schema_df.columns
]
if missing_schema_columns:
    raise RuntimeError(
        "Notebook 02 schema is missing required Global Review retrieval columns: "
        f"{missing_schema_columns}"
    )

if schema_df["parent_asin"].isna().any():
    raise RuntimeError("Schema contains null parent_asin values.")
schema_df["parent_asin"] = schema_df["parent_asin"].astype(str).str.strip()
if schema_df["parent_asin"].eq("").any():
    raise RuntimeError("Schema contains empty parent_asin values.")
if schema_df["parent_asin"].duplicated().any():
    raise RuntimeError("Schema parent_asin must be unique.")

if not schema_df["facet_policy_version"].eq(EXPECTED_FACET_POLICY_VERSION).all():
    raise RuntimeError("Unexpected facet_policy_version.")
if not schema_df["brand_policy"].eq(EXPECTED_BRAND_POLICY).all():
    raise RuntimeError("Unexpected brand_policy.")
if not schema_df["identifier_policy"].eq(EXPECTED_IDENTIFIER_POLICY).all():
    raise RuntimeError("Unexpected identifier_policy.")
if not schema_df["evidence_scope"].eq(PRODUCTION_EVIDENCE_SCOPE).all():
    raise RuntimeError("Notebook 02 evidence_scope must match the Global Review production contract.")
if not schema_df["historical_review_reputation_enabled"].eq(True).all():
    raise RuntimeError("Historical review-reputation must be enabled for the Global Review production pipeline.")
if not schema_df["brand_retrieval_enabled"].eq(True).all():
    raise RuntimeError("brand_retrieval_enabled must be True.")
if not schema_df["brand_profile_enabled"].eq(True).all():
    raise RuntimeError("brand_profile_enabled must be True.")
if not schema_df["brand_query_enabled"].eq(False).all():
    raise RuntimeError("brand_query_enabled must be False.")

for column in ["canonical_retrieval_text", "canonical_text_dense", "canonical_text_sparse"]:
    if schema_df[column].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every retained schema row.")

assert_no_leakage_columns(schema_df, "schema_df")

discontinued_mask = pd.Series(False, index=schema_df.index)
for column in ["is_discontinued_norm", "is_discontinued"]:
    if column in schema_df.columns:
        discontinued_mask |= (
            schema_df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .isin({"yes", "true", "1"})
        )

active_items_df = schema_df.loc[~discontinued_mask].copy().reset_index(drop=True)
discontinued_removed_count = int(discontinued_mask.sum())
if active_items_df.empty:
    raise RuntimeError("No active items remain after discontinued filtering.")

print("Rows:", len(active_items_df))
print("Validation: Global Review schema contract passed")


Rows: 77502
Validation: Global Review schema contract passed


In [7]:
# ==== Item Documents ====
core_doc_columns = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "canonical_metadata_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_dedup_seed",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "evidence_scope",
    "historical_review_reputation_enabled",
    *CATEGORY_REQUIRED_FACET_COLS,
]
review_source_columns = [
    "historical_review_reputation_text",
    "review_reputation_concern_text",
    "review_reputation_skin_type_text",
    "review_reputation_benefit_text",
    "review_reputation_ingredient_text",
    "review_reputation_product_type_or_form_texture_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
]
optional_doc_columns = [
    column for column in [
        "is_discontinued_norm",
        "is_discontinued",
        *review_source_columns,
    ]
    if column in active_items_df.columns
]
item_docs = active_items_df[
    list(dict.fromkeys(core_doc_columns + optional_doc_columns))
].copy()

item_docs["brand_facet_text"] = (
    item_docs["brand_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["specific_query_safe_facet_text"] = (
    item_docs["specific_query_safe_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["functional_facet_text"] = (
    item_docs["functional_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["profile_safe_facet_text"] = (
    item_docs["profile_safe_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["generic_anchor_text"] = (
    item_docs["generic_category_anchor_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["generic_utility_text"] = (
    item_docs["generic_utility_token_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["context_dependent_utility_text"] = (
    item_docs["context_dependent_utility_token_text"].fillna("").astype(str).map(normalize_space)
)

review_signal_source = (
    "review_reputation_only_text"
    if "review_reputation_only_text" in item_docs.columns
    else "review_reputation_facet_text"
    if "review_reputation_facet_text" in item_docs.columns
    else None
)
item_docs["review_derived_signal_text"] = (
    item_docs[review_signal_source].fillna("").astype(str).map(normalize_space)
    if review_signal_source
    else ""
)

item_docs["dense_text_core"] = (
    item_docs["canonical_text_dense_core"].fillna("").astype(str).map(normalize_space)
)
item_docs["sparse_text_core"] = (
    item_docs["canonical_text_sparse_core"].fillna("").astype(str).map(normalize_space)
)

item_docs["canonical_retrieval_text"] = (
    item_docs["canonical_retrieval_text"].fillna("").astype(str).map(normalize_space)
)
item_docs["canonical_text_dense"] = (
    item_docs["canonical_text_dense"].fillna("").astype(str).map(normalize_space)
)
item_docs["canonical_text_sparse"] = (
    item_docs["canonical_text_sparse"].fillna("").astype(str).map(normalize_space)
)
item_docs["dense_text"] = item_docs["canonical_text_dense"]
item_docs["sparse_text"] = item_docs["canonical_text_sparse"]
item_docs["dense_text_v2"] = item_docs["dense_text"]
item_docs["bm25_text"] = item_docs["sparse_text"]
item_docs["bm25_text_v2"] = item_docs["sparse_text"]
item_docs["canonical_text"] = item_docs["canonical_retrieval_text"]

if len(item_docs) <= 0:
    raise RuntimeError("Retained item-doc count must be greater than zero.")
for column in ["dense_text", "sparse_text"]:
    if item_docs[column].eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every retained item.")
if not item_docs["review_derived_signal_text"].fillna("").astype(str).str.strip().ne("").any():
    raise RuntimeError("Global Review item documents require non-empty historical review signals.")
for alias_column, source_column in {
    "dense_text_v2": "dense_text",
    "bm25_text": "sparse_text",
    "bm25_text_v2": "sparse_text",
}.items():
    if not item_docs[alias_column].equals(item_docs[source_column]):
        raise RuntimeError(f"{alias_column} must equal {source_column}.")

brand_in_functional_text_rows = int(
    item_docs.apply(
        lambda row: bool(
            {normalize_entity_value(value) for value in split_values(row.get("brand_facet_text", ""))}
            & {normalize_entity_value(value) for value in split_values(row.get("functional_facet_text", ""))}
        ),
        axis=1,
    ).sum()
)
if brand_in_functional_text_rows:
    raise RuntimeError("functional_facet_text contains brand values.")

def contains_all_brand_values(container_text, brand_text):
    brand_values = {
        normalize_entity_value(value)
        for value in split_values(brand_text)
        if normalize_entity_value(value)
    }
    normalized_container = normalize_entity_value(container_text)
    return all(value in normalized_container for value in brand_values)


brand_rows_mask = item_docs["brand_facet_text"].fillna("").astype(str).str.strip().ne("")
brand_missing_dense_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("dense_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
brand_missing_sparse_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("sparse_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
brand_missing_profile_rows = int(
    item_docs.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("profile_safe_facet_text", ""),
            row.get("brand_facet_text", ""),
        ),
        axis=1,
    ).sum()
)
if brand_missing_dense_rows or brand_missing_sparse_rows:
    raise RuntimeError(
        "Brand must be present in dense and sparse production item text. "
        f"Dense missing: {brand_missing_dense_rows}; sparse missing: {brand_missing_sparse_rows}."
    )
if brand_missing_profile_rows:
    raise RuntimeError(
        f"Brand must be present in profile_safe_facet_text. Missing rows: {brand_missing_profile_rows}."
    )

item_docs_dense = item_docs[[
    "parent_asin",
    "title",
    "dense_text",
    "dense_text_v2",
    "dense_text_core",
    "canonical_text_dense",
    "canonical_retrieval_text_core",
    "canonical_retrieval_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "functional_facet_text",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
    "specific_query_safe_facet_text",
    "review_derived_signal_text",
    "profile_source_text_dedup_seed",
]].copy()

item_docs_sparse = item_docs[[
    "parent_asin",
    "title",
    "sparse_text",
    "bm25_text",
    "bm25_text_v2",
    "sparse_text_core",
    "canonical_text_sparse",
    "canonical_retrieval_text_core",
    "canonical_retrieval_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "functional_facet_text",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
    "specific_query_safe_facet_text",
    "review_derived_signal_text",
    "profile_source_text_dedup_seed",
]].copy()

for frame_name, frame in [
    ("item_docs", item_docs),
    ("item_docs_dense", item_docs_dense),
    ("item_docs_sparse", item_docs_sparse),
]:
    assert_no_leakage_columns(frame, frame_name)

print("Rows:", len(item_docs))
print("Validation: Global Review item documents passed")


Rows: 77502
Validation: Global Review item documents passed


In [8]:
# ==== Item Facets ====
METADATA_FACET_SPECS = [
    ("facet_brand_text", "brand", "HAS_BRAND"),
    ("facet_category_text", "category", "HAS_CATEGORY"),
    ("facet_product_type_text", "product_type", "HAS_PRODUCT_TYPE"),
    ("facet_form_text", "form", "HAS_FORM"),
    ("facet_formulation_text", "formulation", "HAS_FORMULATION"),
    ("facet_texture_text", "texture", "HAS_TEXTURE"),
    ("facet_skin_type_text", "skin_type", "HAS_SKIN_TYPE"),
    ("facet_ingredient_text", "ingredient", "HAS_INGREDIENT"),
    ("facet_concern_text", "concern", "HAS_CONCERN"),
    ("facet_benefit_text", "benefit", "HAS_BENEFIT"),
    ("facet_usage_target_text", "usage_target", "HAS_USAGE_TARGET"),
    ("facet_scent_text", "scent", "HAS_SCENT"),
    ("facet_claim_text", "claim", "HAS_CLAIM"),
]

REVIEW_REPUTATION_FACET_SPECS = [
    ("review_reputation_concern_text", "review_reputation_concern", "HAS_REVIEW_REPUTATION_CONCERN"),
    ("review_reputation_skin_type_text", "review_reputation_skin_type", "HAS_REVIEW_REPUTATION_SKIN_TYPE"),
    ("review_reputation_benefit_text", "review_reputation_benefit", "HAS_REVIEW_REPUTATION_BENEFIT"),
    ("review_reputation_ingredient_text", "review_reputation_ingredient", "HAS_REVIEW_REPUTATION_INGREDIENT"),
    ("review_reputation_product_type_or_form_texture_text", "review_reputation_product_form_texture", "HAS_REVIEW_REPUTATION_PRODUCT_FORM_TEXTURE"),
    ("review_reputation_product_type_text", "review_reputation_product_type", "HAS_REVIEW_REPUTATION_PRODUCT_TYPE"),
    ("review_reputation_form_text", "review_reputation_form", "HAS_REVIEW_REPUTATION_FORM"),
    ("review_reputation_formulation_text", "review_reputation_formulation", "HAS_REVIEW_REPUTATION_FORMULATION"),
    ("review_reputation_texture_text", "review_reputation_texture", "HAS_REVIEW_REPUTATION_TEXTURE"),
    ("review_reputation_usage_target_text", "review_reputation_usage_target", "HAS_REVIEW_REPUTATION_USAGE_TARGET"),
    ("historical_review_concern_text", "review_reputation_concern", "HAS_REVIEW_REPUTATION_CONCERN"),
    ("historical_review_skin_type_text", "review_reputation_skin_type", "HAS_REVIEW_REPUTATION_SKIN_TYPE"),
    ("historical_review_ingredient_text", "review_reputation_ingredient", "HAS_REVIEW_REPUTATION_INGREDIENT"),
    ("historical_review_benefit_text", "review_reputation_benefit", "HAS_REVIEW_REPUTATION_BENEFIT"),
    ("historical_review_form_text", "review_reputation_form", "HAS_REVIEW_REPUTATION_FORM"),
    ("historical_review_texture_text", "review_reputation_texture", "HAS_REVIEW_REPUTATION_TEXTURE"),
    ("historical_review_product_type_text", "review_reputation_product_type", "HAS_REVIEW_REPUTATION_PRODUCT_TYPE"),
]

metadata_specs_used = list(dict.fromkeys([
    spec for spec in METADATA_FACET_SPECS + CATEGORY_OPTIONAL_FACET_SPECS
    if spec[0] in active_items_df.columns
]))
review_specs_used = [
    spec for spec in REVIEW_REPUTATION_FACET_SPECS
    if spec[0] in active_items_df.columns
]
if not review_specs_used and "review_reputation_facet_text" in active_items_df.columns:
    review_specs_used = [
        ("review_reputation_facet_text", "review_reputation", "HAS_REVIEW_REPUTATION")
    ]

FACET_SPECS = metadata_specs_used + review_specs_used
facet_rows = []
for row in active_items_df.itertuples(index=False):
    row_dict = row._asdict()
    parent_asin = normalize_space(row_dict.get("parent_asin"))
    title = normalize_space(row_dict.get("title"))

    for source_column, facet_type, edge_type in FACET_SPECS:
        for value in split_values(row_dict.get(source_column)):
            value_norm = normalize_entity_value(value)
            if value_norm:
                facet_rows.append({
                    "parent_asin": parent_asin,
                    "title": title,
                    "source_column": source_column,
                    "facet_type": facet_type,
                    "facet_value": value,
                    "facet_value_norm": value_norm,
                    "entity_node_id": make_entity_node_id(facet_type, value),
                    "edge_type": edge_type,
                })

item_facets = pd.DataFrame(facet_rows)
if item_facets.empty:
    raise RuntimeError("No item facets were generated.")

item_facets = (
    item_facets
    .drop_duplicates(["parent_asin", "facet_type", "facet_value_norm", "source_column"])
    .sort_values(["parent_asin", "facet_type", "facet_value_norm", "source_column"])
    .reset_index(drop=True)
)
assert_no_leakage_columns(item_facets, "item_facets")


In [9]:
# ==== Facet Flags and Global Review Mask ====
def common_role_from_facet_type(facet_type):
    facet_type = normalize_space(facet_type).lower()
    if facet_type in COMMON_FACET_TYPE_TO_ROLE:
        return COMMON_FACET_TYPE_TO_ROLE[facet_type]
    if facet_type.startswith("review_reputation") or facet_type.startswith("historical_review"):
        return "review_derived_signal"
    return "category_specific_other"

metadata_source_columns = {spec[0] for spec in metadata_specs_used}
item_facets["facet_role"] = item_facets["facet_type"].map(common_role_from_facet_type)
item_facets["facet_family"] = item_facets["facet_role"]

brand_values_by_item = {
    row.parent_asin: {
        normalize_entity_value(value)
        for value in split_values(row.brand_facet_text)
        if normalize_entity_value(value)
    }
    for row in item_docs[["parent_asin", "brand_facet_text"]].itertuples(index=False)
}
item_facets["is_row_brand_value"] = item_facets.apply(
    lambda row: (
        row["facet_value_norm"]
        in brand_values_by_item.get(row["parent_asin"], set())
    ),
    axis=1,
)
item_facets["is_brand"] = (
    item_facets["facet_role"].eq("brand")
    | item_facets["is_row_brand_value"]
)
item_facets["is_review_derived"] = item_facets["facet_role"].eq("review_derived_signal")
item_facets["is_metadata_facet_source"] = item_facets["source_column"].isin(metadata_source_columns)
item_facets["is_disallowed_nonfacet_source"] = item_facets["source_column"].fillna("").astype(str).str.lower().map(
    lambda value: any(term in value for term in FORBIDDEN_FACET_SOURCE_TERMS)
)
item_facets["is_generic_category_anchor"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_GENERIC_CATEGORY_ANCHORS)
)
item_facets["is_generic_utility_token"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_GENERIC_UTILITY_TOKENS)
)
item_facets["is_context_dependent_utility_token"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_CONTEXT_DEPENDENT_UTILITY_TOKENS)
)
item_facets["is_specific_phrase_exception"] = (
    item_facets["facet_value_norm"].fillna("").astype(str).str.lower().isin(COMMON_SPECIFIC_PHRASE_EXCEPTIONS)
)
item_facets["is_specific_facet_phrase"] = (
    item_facets["facet_role"].isin(COMMON_PRODUCT_FUNCTIONAL_ROLES)
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & (
        item_facets["is_specific_phrase_exception"]
        | (
            ~item_facets["is_generic_category_anchor"]
            & ~item_facets["is_generic_utility_token"]
            & ~item_facets["is_context_dependent_utility_token"]
        )
    )
)
item_facets["is_query_safe"] = (
    item_facets["is_specific_facet_phrase"]
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
)
item_facets["is_product_functional_facet"] = (
    item_facets["is_specific_facet_phrase"]
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
)
item_facets["is_core_graph_facet"] = (
    item_facets["is_product_functional_facet"]
    & item_facets["is_query_safe"]
    & ~item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & ~item_facets["is_generic_category_anchor"]
    & ~item_facets["is_generic_utility_token"]
    & ~item_facets["is_context_dependent_utility_token"]
    & item_facets["is_metadata_facet_source"]
    & ~item_facets["is_disallowed_nonfacet_source"]
)
item_facets["is_brand_graph_facet"] = (
    item_facets["facet_role"].eq("brand")
    & item_facets["is_brand"]
    & ~item_facets["is_review_derived"]
    & ~item_facets["is_generic_category_anchor"]
    & ~item_facets["is_generic_utility_token"]
    & ~item_facets["is_context_dependent_utility_token"]
    & item_facets["is_metadata_facet_source"]
    & ~item_facets["is_disallowed_nonfacet_source"]
)
item_facets["is_retrieval_safe"] = (
    item_facets["is_core_graph_facet"]
    | item_facets["is_brand_graph_facet"]
    | (
        item_facets["is_review_derived"]
        & ~item_facets["is_brand"]
        & ~item_facets["is_generic_category_anchor"]
        & ~item_facets["is_generic_utility_token"]
        & ~item_facets["is_context_dependent_utility_token"]
        & ~item_facets["is_disallowed_nonfacet_source"]
    )
)
item_facets["is_profile_safe"] = (
    item_facets["is_core_graph_facet"]
    | item_facets["is_brand_graph_facet"]
)
item_facets["is_global_review_graph_facet"] = item_facets["is_retrieval_safe"]
item_facets["common_framework_version"] = COMMON_RETRIEVAL_ARTIFACT_FRAMEWORK_VERSION

core_item_facets = item_facets.loc[item_facets["is_core_graph_facet"]].copy().reset_index(drop=True)
brand_item_facets = item_facets.loc[item_facets["is_brand_graph_facet"]].copy().reset_index(drop=True)
global_review_item_facets = item_facets.loc[
    item_facets["is_global_review_graph_facet"]
].copy().reset_index(drop=True)
review_item_facets = global_review_item_facets.loc[
    global_review_item_facets["is_review_derived"]
].copy().reset_index(drop=True)

if core_item_facets.empty:
    raise RuntimeError("Metadata product-functional facet rows must be greater than zero.")
if brand_item_facets.empty:
    raise RuntimeError("Brand facet rows must be greater than zero.")
if review_item_facets.empty:
    raise RuntimeError("Global Review requires non-empty historical review facet rows.")

bad_brand_functional_rows = item_facets[item_facets["is_brand"] & item_facets["is_product_functional_facet"]]
if len(bad_brand_functional_rows):
    display(bad_brand_functional_rows[["parent_asin", "facet_type", "facet_role", "facet_value_norm", "source_column"]].head(50))
    raise RuntimeError("Brand rows must not be product-functional facets.")
bad_brand_query_safe_rows = item_facets[item_facets["is_brand"] & item_facets["is_query_safe"]]
if len(bad_brand_query_safe_rows):
    display(bad_brand_query_safe_rows[["parent_asin", "facet_type", "facet_role", "facet_value_norm", "source_column"]].head(50))
    raise RuntimeError("Brand rows must not be query-safe.")
bad_brand_core_graph_rows = item_facets[item_facets["is_brand"] & item_facets["is_core_graph_facet"]]
if len(bad_brand_core_graph_rows):
    display(bad_brand_core_graph_rows[["parent_asin", "facet_type", "facet_role", "facet_value_norm", "source_column"]].head(50))
    raise RuntimeError("Brand rows must not enter the functional/core graph.")
if not item_facets.loc[item_facets["is_brand_graph_facet"], "is_retrieval_safe"].all():
    raise RuntimeError("Every brand graph facet must be retrieval-safe.")
if not item_facets.loc[item_facets["is_brand_graph_facet"], "is_profile_safe"].all():
    raise RuntimeError("Every brand graph facet must be profile-safe.")
if item_facets.loc[item_facets["is_brand_graph_facet"], "is_product_functional_facet"].any():
    raise RuntimeError("Brand graph facets must remain separate from product-functional facets.")
if item_facets.loc[item_facets["is_review_derived"], "is_product_functional_facet"].any():
    raise RuntimeError("Review-derived rows must not be product-functional facets.")
if item_facets.loc[item_facets["is_review_derived"], "is_query_safe"].any():
    raise RuntimeError("Review-derived rows must not be query-safe.")
if item_facets.loc[item_facets["is_generic_category_anchor"], "is_specific_facet_phrase"].any():
    raise RuntimeError("Generic category anchors must not be specific facet phrases.")
if item_facets.loc[item_facets["is_generic_utility_token"], "is_specific_facet_phrase"].any():
    raise RuntimeError("Standalone generic utility tokens must not be specific facet phrases.")
if item_facets.loc[
    item_facets["is_context_dependent_utility_token"] & ~item_facets["is_specific_phrase_exception"],
    "is_specific_facet_phrase",
].any():
    raise RuntimeError("Standalone context-dependent utility tokens must not be specific facet phrases.")

facet_counts = (
    item_facets.groupby("facet_type").size().sort_values(ascending=False).astype(int).to_dict()
)
facet_vocab = {
    facet_type: sorted(values)
    for facet_type, values in (
        global_review_item_facets
        .groupby("facet_type")["facet_value_norm"]
        .apply(lambda series: set(series.dropna().astype(str)))
        .items()
    )
}
facet_vocab_df = (
    global_review_item_facets[["facet_type", "facet_value_norm"]]
    .drop_duplicates()
    .sort_values(["facet_type", "facet_value_norm"])
    .reset_index(drop=True)
)

facet_role_coverage_df = (
    item_facets
    .groupby(["facet_role", "facet_type"], dropna=False)
    .agg(
        facet_rows=("facet_value_norm", "size"),
        item_count=("parent_asin", "nunique"),
        unique_value_count=("facet_value_norm", "nunique"),
        brand_rows=("is_brand", "sum"),
        row_brand_value_rows=("is_row_brand_value", "sum"),
        query_safe_rows=("is_query_safe", "sum"),
        product_functional_facet_rows=("is_product_functional_facet", "sum"),
        review_derived_rows=("is_review_derived", "sum"),
        generic_anchor_rows=("is_generic_category_anchor", "sum"),
        generic_utility_rows=("is_generic_utility_token", "sum"),
        context_dependent_utility_rows=("is_context_dependent_utility_token", "sum"),
        specific_facet_phrase_rows=("is_specific_facet_phrase", "sum"),
        core_graph_rows=("is_core_graph_facet", "sum"),
        brand_graph_rows=("is_brand_graph_facet", "sum"),
        retrieval_safe_rows=("is_retrieval_safe", "sum"),
        profile_safe_rows=("is_profile_safe", "sum"),
        global_review_graph_rows=("is_global_review_graph_facet", "sum"),
    )
    .reset_index()
    .sort_values(["facet_role", "facet_rows"], ascending=[True, False])
)
facet_role_coverage_df["with_brand_item_coverage_rate"] = (
    facet_role_coverage_df["item_count"] / max(active_items_df["parent_asin"].nunique(), 1)
)
facet_role_coverage_df["no_brand_item_coverage_rate"] = np.where(
    facet_role_coverage_df["facet_role"].eq("brand"),
    0.0,
    facet_role_coverage_df["with_brand_item_coverage_rate"],
)

print("Rows:", len(item_facets))
print("Rows:", len(core_item_facets))
print("Rows:", len(review_item_facets))
print("Rows:", len(global_review_item_facets))
print("Validation: Global Review facet mask passed")


Rows: 984052
Rows: 618684
Rows: 284282
Rows: 977717
Validation: Global Review facet mask passed


In [10]:
# ==== Global Review Item Graph ====
graph_long = global_review_item_facets[[
    "parent_asin",
    "title",
    "entity_node_id",
    "edge_type",
    "facet_type",
    "facet_value",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_query_safe",
    "is_product_functional_facet",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()

graph_long["product_node_id"] = "item::" + graph_long["parent_asin"].astype(str)
graph_long["source_node_id"] = graph_long["product_node_id"]
graph_long["target_node_id"] = graph_long["entity_node_id"]
graph_long["source_node_type"] = "item"
graph_long["target_node_type"] = "entity"

graph_long = graph_long[[
    "parent_asin",
    "title",
    "product_node_id",
    "entity_node_id",
    "source_node_id",
    "target_node_id",
    "source_node_type",
    "target_node_type",
    "edge_type",
    "facet_type",
    "facet_value",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_query_safe",
    "is_product_functional_facet",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].drop_duplicates().reset_index(drop=True)

graph_edges = graph_long[[
    "source_node_id",
    "target_node_id",
    "source_node_type",
    "target_node_type",
    "edge_type",
    "facet_type",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_query_safe",
    "is_product_functional_facet",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()
graph_edges["edge_id"] = (
    graph_edges["source_node_id"].astype(str)
    + "||"
    + graph_edges["edge_type"].astype(str)
    + "||"
    + graph_edges["target_node_id"].astype(str)
)
graph_edges = graph_edges.drop_duplicates("edge_id").reset_index(drop=True)

item_kg_edges = graph_edges.rename(
    columns={"source_node_id": "product_node_id", "target_node_id": "entity_node_id"}
)[[
    "edge_id",
    "product_node_id",
    "entity_node_id",
    "edge_type",
    "facet_type",
    "facet_value_norm",
    "source_column",
    "facet_role",
    "facet_family",
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_query_safe",
    "is_product_functional_facet",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]].copy()

product_node_cols = [
    "parent_asin",
    "title",
    "brand_facet_text",
    "profile_safe_facet_text",
    "query_safe_facet_text",
    "functional_facet_text",
]
product_nodes = item_docs[product_node_cols].copy()
product_nodes["node_id"] = "item::" + product_nodes["parent_asin"].astype(str)
product_nodes["node_type"] = "item"
product_nodes = product_nodes[["node_id", "node_type"] + product_node_cols].drop_duplicates("node_id").reset_index(drop=True)

entity_nodes = graph_long[["entity_node_id", "facet_type", "facet_value_norm"]].drop_duplicates().copy()
entity_nodes = entity_nodes.rename(columns={"entity_node_id": "node_id"})
entity_nodes["node_type"] = "entity"
entity_nodes = entity_nodes[["node_id", "node_type", "facet_type", "facet_value_norm"]].sort_values("node_id").reset_index(drop=True)

for frame_name, frame in [
    ("graph_long", graph_long),
    ("graph_edges", graph_edges),
    ("item_kg_edges", item_kg_edges),
    ("product_nodes", product_nodes),
    ("entity_nodes", entity_nodes),
]:
    assert_no_leakage_columns(frame, frame_name)

metadata_graph_edges = int(graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum())
brand_graph_edges = int(graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum())
review_reputation_graph_edges = int(graph_edges["is_review_derived"].fillna(False).astype(bool).sum())
total_graph_edges = int(len(graph_edges))
row_brand_value_graph_edges = int(graph_edges["is_row_brand_value"].fillna(False).astype(bool).sum())
generic_anchor_graph_edges = int(graph_edges["is_generic_category_anchor"].fillna(False).astype(bool).sum())
generic_utility_graph_edges = int(graph_edges["is_generic_utility_token"].fillna(False).astype(bool).sum())
context_utility_graph_edges = int(graph_edges["is_context_dependent_utility_token"].fillna(False).astype(bool).sum())
disallowed_source_graph_edges = int(graph_edges["is_disallowed_nonfacet_source"].fillna(False).astype(bool).sum())

if metadata_graph_edges <= 0:
    raise RuntimeError("Metadata graph edges must be greater than zero.")
if brand_graph_edges <= 0:
    raise RuntimeError("Brand graph edges must be greater than zero.")
if review_reputation_graph_edges <= 0:
    raise RuntimeError("Global Review requires historical review-reputation graph edges.")
if total_graph_edges != metadata_graph_edges + brand_graph_edges + review_reputation_graph_edges:
    raise RuntimeError(
        "Global Review graph must contain only metadata functional, brand, and review-derived edges."
    )
if row_brand_value_graph_edges != brand_graph_edges:
    raise RuntimeError(
        "Every production brand edge must come from the explicit brand facet channel."
    )
if generic_anchor_graph_edges or generic_utility_graph_edges:
    raise RuntimeError("Generic anchors or utility tokens must not enter the Global Review graph.")
if context_utility_graph_edges or disallowed_source_graph_edges:
    raise RuntimeError("Context utility or disallowed source rows must not enter the Global Review graph.")

metadata_edge_mask = graph_edges["is_core_graph_facet"].fillna(False).astype(bool)
brand_edge_mask = graph_edges["is_brand_graph_facet"].fillna(False).astype(bool)
review_edge_mask = graph_edges["is_review_derived"].fillna(False).astype(bool)
if not graph_edges.loc[metadata_edge_mask, "is_product_functional_facet"].all():
    raise RuntimeError("Every metadata graph edge must be product-functional.")
if not graph_edges.loc[metadata_edge_mask, "is_query_safe"].all():
    raise RuntimeError("Every metadata graph edge must be query-safe.")
if not graph_edges.loc[metadata_edge_mask, "is_metadata_facet_source"].all():
    raise RuntimeError("Every metadata graph edge must come from metadata facets.")
bad_brand_functional_graph_edges = graph_edges[
    brand_edge_mask & graph_edges["is_product_functional_facet"].fillna(False).astype(bool)
]
if len(bad_brand_functional_graph_edges):
    display(
        bad_brand_functional_graph_edges[
            ["item_id", "entity_id", "facet_type", "facet_role", "facet_value_norm", "source_column"]
        ].head(50)
    )
    raise RuntimeError("Brand graph edges must remain separate from product-functional facets.")
bad_brand_core_graph_edges = graph_edges[
    brand_edge_mask & graph_edges["is_core_graph_facet"].fillna(False).astype(bool)
]
if len(bad_brand_core_graph_edges):
    display(
        bad_brand_core_graph_edges[
            ["item_id", "entity_id", "facet_type", "facet_role", "facet_value_norm", "source_column"]
        ].head(50)
    )
    raise RuntimeError("Brand graph edges must not enter the functional/core graph.")
if graph_edges.loc[brand_edge_mask, "is_query_safe"].any():
    raise RuntimeError("Brand graph edges must not be query-safe.")
if not graph_edges.loc[brand_edge_mask, "is_profile_safe"].all():
    raise RuntimeError("Brand graph edges must be profile-safe.")
if not graph_edges.loc[brand_edge_mask, "is_retrieval_safe"].all():
    raise RuntimeError("Brand graph edges must be retrieval-safe.")
if graph_edges.loc[review_edge_mask, "is_product_functional_facet"].any():
    raise RuntimeError("Review-derived graph edges must remain separate from product-functional facets.")
if graph_edges.loc[review_edge_mask, "is_query_safe"].any():
    raise RuntimeError("Review-derived graph edges must not be marked query-safe.")

core_facet_filter_audit_df = pd.DataFrame([{
    "category_id": "face",
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "all_facet_rows": int(len(item_facets)),
    "core_facet_rows": int(len(core_item_facets)),
    "brand_facet_rows": int(len(brand_item_facets)),
    "review_facet_rows": int(len(review_item_facets)),
    "global_review_facet_rows": int(len(global_review_item_facets)),
    "core_facet_items": int(core_item_facets["parent_asin"].nunique()),
    "metadata_functional_facet_rows": int(len(core_item_facets)),
    "metadata_graph_edges": metadata_graph_edges,
    "brand_graph_edges": brand_graph_edges,
    "review_reputation_graph_edges": review_reputation_graph_edges,
    "global_review_graph_edges": total_graph_edges,
    "review_derived_rows_in_core_mask": int(core_item_facets["is_review_derived"].sum()),
    "row_brand_value_rows_in_core_mask": int(core_item_facets["is_row_brand_value"].sum()),
    "row_brand_value_graph_edges": row_brand_value_graph_edges,
    "brand_rows_in_core_mask": int(core_item_facets["is_brand"].sum()),
    "generic_anchor_rows_in_core_mask": int(core_item_facets["is_generic_category_anchor"].sum()),
    "generic_anchor_graph_edges": generic_anchor_graph_edges,
    "generic_utility_rows_in_core_mask": int(core_item_facets["is_generic_utility_token"].sum()),
    "generic_utility_graph_edges": generic_utility_graph_edges,
    "context_utility_graph_edges": context_utility_graph_edges,
    "disallowed_source_rows_in_core_mask": int(core_item_facets["is_disallowed_nonfacet_source"].sum()),
    "disallowed_source_graph_edges": disallowed_source_graph_edges,
    "status": "PASS",
}])

print("Rows:", len(graph_edges))
print("Validation: Global Review graph passed")


Rows: 977717
Validation: Global Review graph passed


In [11]:
# ==== Debug: Global Review Retrieval Artifact Contract Checks ====
def _debug_count_true(series):
    return int(series.fillna(False).astype(bool).sum())


dense_alias_mismatch = int(
    (
        item_docs["dense_text"].fillna("").astype(str) != item_docs["canonical_text_dense"].fillna("").astype(str)
    ).sum()
)
sparse_alias_mismatch = int(
    (
        item_docs["sparse_text"].fillna("").astype(str) != item_docs["canonical_text_sparse"].fillna("").astype(str)
    ).sum()
)
metadata_functional_facet_rows = int(len(core_item_facets))
review_facet_rows = int(len(review_item_facets))
metadata_graph_edges_debug = int(
    graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum()
)
review_graph_edges_debug = _debug_count_true(graph_edges["is_review_derived"])
brand_graph_rows = _debug_count_true(graph_edges["is_brand_graph_facet"])
row_brand_graph_rows = _debug_count_true(graph_edges["is_row_brand_value"])

debug_checks = pd.DataFrame([
    {
        "check": "dense_text == canonical_text_dense",
        "value": dense_alias_mismatch,
        "expected": 0,
        "pass": dense_alias_mismatch == 0,
    },
    {
        "check": "sparse_text == canonical_text_sparse",
        "value": sparse_alias_mismatch,
        "expected": 0,
        "pass": sparse_alias_mismatch == 0,
    },
    {
        "check": "metadata functional facet rows > 0",
        "value": metadata_functional_facet_rows,
        "expected": "> 0",
        "pass": metadata_functional_facet_rows > 0,
    },
    {
        "check": "historical review facet rows > 0",
        "value": review_facet_rows,
        "expected": "> 0",
        "pass": review_facet_rows > 0,
    },
    {
        "check": "metadata graph edges > 0",
        "value": metadata_graph_edges_debug,
        "expected": "> 0",
        "pass": metadata_graph_edges_debug > 0,
    },
    {
        "check": "review-derived graph edges > 0",
        "value": review_graph_edges_debug,
        "expected": "> 0",
        "pass": review_graph_edges_debug > 0,
    },
    {
        "check": "brand graph rows > 0",
        "value": brand_graph_rows,
        "expected": "> 0",
        "pass": brand_graph_rows > 0,
    },
    {
        "check": "row-level brand graph rows equal explicit brand graph rows",
        "value": row_brand_graph_rows,
        "expected": brand_graph_rows,
        "pass": row_brand_graph_rows == brand_graph_rows,
    },
])

display(debug_checks)

failed_checks = debug_checks.loc[~debug_checks["pass"], "check"].tolist()
if failed_checks:
    raise RuntimeError(f"Global Review retrieval artifact checks failed: {failed_checks}")

print("Debug checks passed.")


,check,value,expected,pass
0,dense_text == canonical_text_dense,0,0,True
1,sparse_text == canonical_text_sparse,0,0,True
2,metadata functional facet rows > 0,618684,> 0,True
3,historical review facet rows > 0,284282,> 0,True
4,metadata graph edges > 0,618684,> 0,True
5,review-derived graph edges > 0,284282,> 0,True
6,brand graph rows > 0,74751,> 0,True
7,row-level brand graph rows equal explicit brand graph rows,74751,74751,True


Debug checks passed.


In [12]:
# ==== Temporal Artifact Precompute ====
MS_PER_DAY = 86_400_000

ITEM_ID_COLUMN_CANDIDATES = [
    "parent_asin",
    "parentAsin",
    "parent_asin_review",
    "item_parent_asin",
    "asin",
]

USER_ID_COLUMN_CANDIDATES = [
    "user_id",
    "reviewer_id",
    "reviewerID",
    "customer_id",
]

TIMESTAMP_COLUMN_CANDIDATES = [
    "review_timestamp_ms",
    "review_timestamp",
    "timestamp_ms",
    "timestamp",
    "unixReviewTime",
    "unix_review_time",
    "review_time",
    "review_date",
    "date",
]

RAW_REVIEW_TEXT_COLUMNS = {
    "text",
    "review_text",
    "review_body",
    "raw_review_text",
    "target_review_text",
    "heldout_review_text",
    "review_title",
    "review_summary",
    "summary",
    "body",
}


def assert_no_raw_review_text_columns(df, frame_name):
    blocked = sorted(set(map(str, df.columns)) & RAW_REVIEW_TEXT_COLUMNS)
    blocked += [
        str(col)
        for col in df.columns
        if str(col).startswith(("review_text_", "review_body_", "raw_review_", "target_review_", "heldout_review_"))
    ]
    blocked = sorted(set(blocked))
    if blocked:
        raise RuntimeError(f"{frame_name} contains raw review text columns: {blocked}")


def get_table_columns(path):
    path = Path(path)
    name = path.name.lower()

    if name.endswith(".parquet"):
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(path).schema.names)

    if name.endswith(".csv") or name.endswith(".csv.gz"):
        return list(pd.read_csv(path, nrows=0).columns)

    if name.endswith(".tsv") or name.endswith(".tsv.gz"):
        return list(pd.read_csv(path, nrows=0, sep="\t").columns)

    return []


def read_metadata_columns(path, columns):
    path = Path(path)
    name = path.name.lower()

    if name.endswith(".parquet"):
        return pd.read_parquet(path, columns=columns)

    if name.endswith(".csv") or name.endswith(".csv.gz"):
        return pd.read_csv(path, usecols=columns)

    if name.endswith(".tsv") or name.endswith(".tsv.gz"):
        return pd.read_csv(path, usecols=columns, sep="\t")

    raise RuntimeError(f"Unsupported review metadata file type: {path}")


def first_existing(columns, candidates):
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None


def review_metadata_source_path():
    return RAW_REVIEWS_PATH

def coerce_review_timestamp_ms(series):
    numeric = pd.to_numeric(series, errors="coerce")
    source_non_null = int(series.notna().sum())
    numeric_non_null = int(numeric.notna().sum())

    if source_non_null > 0 and numeric_non_null / source_non_null >= 0.95:
        median_value = float(numeric.dropna().median())
        if median_value < 100_000_000_000:
            numeric = numeric * 1000
        return numeric.round().astype("Int64")

    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    out = pd.Series(pd.NA, index=series.index, dtype="Int64")
    valid_mask = parsed.notna()
    out.loc[valid_mask] = (parsed.loc[valid_mask].astype("int64") // 1_000_000).astype("int64")
    return out


def normalize_parent_asin_from_review_source(df, item_col):
    item_values = df[item_col].fillna("").astype(str).str.strip()

    if item_col == "asin" and "asin" in schema_df.columns:
        asin_to_parent = (
            schema_df[["asin", "parent_asin"]]
            .dropna()
            .assign(asin=lambda x: x["asin"].astype(str).str.strip())
            .drop_duplicates("asin")
        )
        mapped = pd.DataFrame({"asin": item_values}).merge(asin_to_parent, on="asin", how="left")
        return mapped["parent_asin"].fillna("").astype(str).str.strip()

    return item_values


def load_review_metadata_for_temporal_artifacts():
    active_parent_asins = set(active_items_df["parent_asin"].astype(str))
    path = review_metadata_source_path()
    columns = get_table_columns(path)
    item_col = first_existing(columns, ITEM_ID_COLUMN_CANDIDATES)
    timestamp_col = first_existing(columns, TIMESTAMP_COLUMN_CANDIDATES)
    user_col = first_existing(columns, USER_ID_COLUMN_CANDIDATES)

    if item_col is None or timestamp_col is None:
        raise RuntimeError(f"Review metadata source missing item or timestamp column: {path}")

    usecols = list(dict.fromkeys([item_col, timestamp_col] + ([user_col] if user_col else [])))
    source_df = read_metadata_columns(path, usecols)
    assert_no_raw_review_text_columns(source_df, "review_metadata_source_df")

    review_metadata = pd.DataFrame()
    review_metadata["parent_asin"] = normalize_parent_asin_from_review_source(source_df, item_col)
    review_metadata["review_timestamp_ms"] = coerce_review_timestamp_ms(source_df[timestamp_col])

    if user_col:
        review_metadata["user_id"] = source_df[user_col].fillna("").astype(str).str.strip()

    review_metadata = review_metadata.loc[
        review_metadata["parent_asin"].ne("")
        & review_metadata["review_timestamp_ms"].notna()
        & (review_metadata["review_timestamp_ms"] > 0)
    ].copy()

    review_metadata = review_metadata.loc[
        review_metadata["parent_asin"].astype(str).isin(active_parent_asins)
    ].copy()

    if review_metadata.empty:
        raise RuntimeError(f"Review metadata source has no valid active-item rows: {path}")

    review_metadata["review_day"] = (review_metadata["review_timestamp_ms"] // MS_PER_DAY).astype("Int64")
    review_metadata["review_date"] = pd.to_datetime(
        review_metadata["review_timestamp_ms"].astype("int64"),
        unit="ms",
        utc=True,
    ).dt.strftime("%Y-%m-%d")

    sort_cols = ["parent_asin", "review_timestamp_ms"]
    if "user_id" in review_metadata.columns:
        sort_cols.append("user_id")
    review_metadata = review_metadata.sort_values(sort_cols).reset_index(drop=True)

    source_info = {
        "path": str(path),
        "file_type": path.suffix.lower(),
        "item_column": item_col,
        "timestamp_column": timestamp_col,
        "user_column": user_col,
        "columns_loaded": usecols,
        "raw_review_text_columns_loaded": [],
        "raw_review_text_columns_present_in_source": sorted(set(columns) & RAW_REVIEW_TEXT_COLUMNS),
        "valid_rows": int(len(review_metadata)),
    }
    return review_metadata, source_info


review_metadata_df, review_metadata_source_info = load_review_metadata_for_temporal_artifacts()
assert_no_leakage_columns(review_metadata_df, "review_metadata_df")
assert_no_raw_review_text_columns(review_metadata_df, "review_metadata_df")

index_cols = ["parent_asin"]
if "user_id" in review_metadata_df.columns:
    index_cols.append("user_id")
index_cols += ["review_timestamp_ms", "review_day", "review_date"]

item_review_timestamp_index = review_metadata_df[index_cols].copy()
item_review_timestamp_index["item_review_sequence"] = (
    item_review_timestamp_index
    .groupby("parent_asin")
    .cumcount()
    .add(1)
    .astype("int64")
)

item_daily_review_counts = (
    item_review_timestamp_index
    .groupby(["parent_asin", "review_day", "review_date"], dropna=False)
    .size()
    .reset_index(name="review_count")
    .sort_values(["parent_asin", "review_day"])
    .reset_index(drop=True)
)

facet_daily_source = core_item_facets[[
    "parent_asin",
    "facet_type",
    "facet_value_norm",
    "entity_node_id",
]].drop_duplicates()

entity_daily_review_counts = (
    item_review_timestamp_index[["parent_asin", "review_day", "review_date"]]
    .merge(facet_daily_source, on="parent_asin", how="inner")
    .groupby(["facet_type", "facet_value_norm", "entity_node_id", "review_day", "review_date"], dropna=False)
    .agg(
        review_count=("parent_asin", "size"),
        item_count=("parent_asin", "nunique"),
    )
    .reset_index()
    .sort_values(["facet_type", "facet_value_norm", "review_day"])
    .reset_index(drop=True)
)

item_temporal_summary_base = (
    item_review_timestamp_index
    .groupby("parent_asin")
    .agg(
        item_precomputed_review_count=("review_timestamp_ms", "size"),
        first_review_timestamp_ms=("review_timestamp_ms", "min"),
        last_review_timestamp_ms=("review_timestamp_ms", "max"),
        first_review_day=("review_day", "min"),
        last_review_day=("review_day", "max"),
        active_review_days=("review_day", "nunique"),
    )
    .reset_index()
)

item_temporal_summary = (
    active_items_df[["parent_asin"]]
    .drop_duplicates()
    .merge(item_temporal_summary_base, on="parent_asin", how="left")
)

count_cols = ["item_precomputed_review_count", "active_review_days"]
for col in count_cols:
    item_temporal_summary[col] = item_temporal_summary[col].fillna(0).astype("int64")

timestamp_cols = ["first_review_timestamp_ms", "last_review_timestamp_ms", "first_review_day", "last_review_day"]
for col in timestamp_cols:
    item_temporal_summary[col] = item_temporal_summary[col].astype("Int64")

item_temporal_summary["first_review_date"] = pd.to_datetime(
    item_temporal_summary["first_review_timestamp_ms"], unit="ms", utc=True
).dt.strftime("%Y-%m-%d")
item_temporal_summary["last_review_date"] = pd.to_datetime(
    item_temporal_summary["last_review_timestamp_ms"], unit="ms", utc=True
).dt.strftime("%Y-%m-%d")

for frame_name, frame in [
    ("item_review_timestamp_index", item_review_timestamp_index),
    ("item_daily_review_counts", item_daily_review_counts),
    ("entity_daily_review_counts", entity_daily_review_counts),
    ("item_temporal_summary", item_temporal_summary),
]:
    assert_no_leakage_columns(frame, frame_name)
    assert_no_raw_review_text_columns(frame, frame_name)

if item_review_timestamp_index.empty:
    raise RuntimeError("item_review_timestamp_index must not be empty.")

if item_daily_review_counts.empty:
    raise RuntimeError("item_daily_review_counts must not be empty.")

if entity_daily_review_counts.empty:
    raise RuntimeError("entity_daily_review_counts must not be empty.")

temporal_artifact_outputs = {
    "item_review_timestamp_index": ITEM_REVIEW_TIMESTAMP_INDEX_PATH,
    "item_daily_review_counts": ITEM_DAILY_REVIEW_COUNTS_PATH,
    "entity_daily_review_counts": ENTITY_DAILY_REVIEW_COUNTS_PATH,
    "item_temporal_summary": ITEM_TEMPORAL_SUMMARY_PATH,
    "temporal_artifact_manifest": TEMPORAL_MANIFEST_PATH,
}

temporal_row_counts = {
    "review_metadata_rows_loaded": int(len(review_metadata_df)),
    "item_review_timestamp_index": int(len(item_review_timestamp_index)),
    "item_daily_review_counts": int(len(item_daily_review_counts)),
    "entity_daily_review_counts": int(len(entity_daily_review_counts)),
    "item_temporal_summary": int(len(item_temporal_summary)),
    "items_with_reviews": int(item_review_timestamp_index["parent_asin"].nunique()),
    "entity_facet_types": int(entity_daily_review_counts["facet_type"].nunique()),
}

temporal_facet_types = sorted(entity_daily_review_counts["facet_type"].dropna().astype(str).unique().tolist())


In [13]:
# ==== Output Export ====
active_schema_export_columns = [
    column for column in active_items_df.columns
    if column != "identifier_diagnostic_text"
]
active_items_df[active_schema_export_columns].to_parquet(ACTIVE_SCHEMA_PATH, index=False)
item_docs_dense.to_parquet(ITEM_DOCS_DENSE_PATH, index=False)
item_docs_sparse.to_parquet(ITEM_DOCS_SPARSE_PATH, index=False)
item_docs.to_parquet(ITEM_DOCS_PATH, index=False)
item_facets.to_parquet(ITEMS_FACETS_PATH, index=False)
graph_long.to_parquet(ITEM_GRAPH_LONG_PATH, index=False)
item_kg_edges.to_parquet(ITEM_KG_EDGES_PATH, index=False)
graph_edges.to_parquet(GRAPH_EDGES_PATH, index=False)
product_nodes.to_parquet(PRODUCT_NODES_PATH, index=False)
entity_nodes.to_parquet(ENTITY_NODES_PATH, index=False)
item_review_timestamp_index.to_parquet(ITEM_REVIEW_TIMESTAMP_INDEX_PATH, index=False)
item_daily_review_counts.to_parquet(ITEM_DAILY_REVIEW_COUNTS_PATH, index=False)
entity_daily_review_counts.to_parquet(ENTITY_DAILY_REVIEW_COUNTS_PATH, index=False)
item_temporal_summary.to_parquet(ITEM_TEMPORAL_SUMMARY_PATH, index=False)

with open(FACET_VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(facet_vocab, file, ensure_ascii=False, indent=2)
facet_vocab_df.to_parquet(FACET_VOCAB_TABLE_PATH, index=False)

facet_coverage_df = (
    item_facets.groupby(["facet_role", "facet_type"], dropna=False)
    .agg(
        row_count=("facet_value_norm", "size"),
        item_count=("parent_asin", "nunique"),
        unique_value_count=("facet_value_norm", "nunique"),
        core_graph_rows=("is_core_graph_facet", "sum"),
    )
    .reset_index()
)
facet_coverage_df.to_csv(FACET_COVERAGE_PATH, index=False, encoding="utf-8-sig")

doc_coverage_df = pd.DataFrame([
    {
        "column": column,
        "non_empty_rows": int(item_docs[column].fillna("").astype(str).str.strip().ne("").sum()),
        "coverage_ratio": float(item_docs[column].fillna("").astype(str).str.strip().ne("").mean()),
    }
    for column in item_docs.columns
])
doc_coverage_df.to_csv(DOC_COVERAGE_PATH, index=False, encoding="utf-8-sig")
core_facet_filter_audit_df.to_csv(CORE_FACET_FILTER_AUDIT_PATH, index=False, encoding="utf-8-sig")

temporal_manifest = {
    "review_metadata_source": str(RAW_REVIEWS_PATH),
    "row_counts": temporal_row_counts,
    "leakage_rule": TEMPORAL_LEAKAGE_RULE,
    "entity_facet_source": "core_item_facets",
    "raw_review_text_loaded": False,
    "raw_review_text_exported": False,
    "rating_used": False,
    "sentiment_used": False,
}
with open(TEMPORAL_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(temporal_manifest, file, ensure_ascii=False, indent=2)

print("Output:", ITEM_DOCS_PATH)
print("Output:", ITEMS_FACETS_PATH)
print("Output:", GRAPH_EDGES_PATH)
print("Output:", CORE_FACET_FILTER_AUDIT_PATH)
print("Rows:", len(item_docs))


Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_docs.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_items_facets.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_graph_edges.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_core_facet_filter_audit.csv
Rows: 77502


In [14]:
# ==== Final Validation and Manifest ====
for frame_name, frame in [
    ("item_docs", item_docs),
    ("item_docs_dense", item_docs_dense),
    ("item_docs_sparse", item_docs_sparse),
]:
    if len(frame) <= 0:
        raise RuntimeError(f"{frame_name} must contain retained items.")
    if not frame["parent_asin"].astype(str).is_unique:
        raise RuntimeError(f"{frame_name} parent_asin must be unique.")

required_doc_columns = [
    "dense_text_core",
    "sparse_text_core",
    "dense_text",
    "sparse_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "generic_anchor_text",
    "generic_utility_text",
    "context_dependent_utility_text",
]
missing_doc_columns = [column for column in required_doc_columns if column not in item_docs.columns]
if missing_doc_columns:
    raise RuntimeError(f"Missing required item-doc columns: {missing_doc_columns}")

for column in ["dense_text", "sparse_text"]:
    if item_docs[column].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every retained item.")
if not item_docs["dense_text"].equals(item_docs["canonical_text_dense"]):
    raise RuntimeError("dense_text must equal canonical_text_dense.")
if not item_docs["sparse_text"].equals(item_docs["canonical_text_sparse"]):
    raise RuntimeError("sparse_text must equal canonical_text_sparse.")

required_facet_flags = [
    "is_row_brand_value",
    "is_brand",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_review_derived",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]
missing_facet_flags = [column for column in required_facet_flags if column not in item_facets.columns]
if missing_facet_flags:
    raise RuntimeError(f"Missing item-facet flags: {missing_facet_flags}")

if len(core_item_facets) <= 0:
    raise RuntimeError("Metadata functional facet rows must be greater than zero.")
if len(brand_item_facets) <= 0:
    raise RuntimeError("Brand facet rows must be greater than zero.")
if len(review_item_facets) <= 0:
    raise RuntimeError("Historical review facet rows must be greater than zero.")
if len(graph_edges) <= 0:
    raise RuntimeError("Global Review graph edges must be greater than zero.")
for column in [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_disallowed_nonfacet_source",
]:
    if graph_edges[column].fillna(False).astype(bool).any():
        raise RuntimeError(f"Production graph contains disallowed rows: {column}")
if not graph_edges["is_global_review_graph_facet"].all():
    raise RuntimeError("Every production graph edge must satisfy the Global Review graph mask.")
if not graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).any():
    raise RuntimeError("Production Global Review graph must include brand edges.")
if graph_edges.loc[graph_edges["is_brand_graph_facet"], "is_query_safe"].any():
    raise RuntimeError("Brand graph edges must not be query-safe.")
if not graph_edges.loc[graph_edges["is_brand_graph_facet"], "is_profile_safe"].all():
    raise RuntimeError("Brand graph edges must be profile-safe.")
if not graph_edges["is_review_derived"].fillna(False).astype(bool).any():
    raise RuntimeError("Production Global Review graph must include historical review-derived edges.")

expected_output_names = {
    ITEM_DOCS_PATH: "face_item_docs.parquet",
    ITEMS_FACETS_PATH: "face_items_facets.parquet",
    FACET_VOCAB_PATH: "face_facet_vocab.json",
    GRAPH_EDGES_PATH: "face_item_graph_edges.parquet",
    MANIFEST_PATH: "retrieval_artifact_manifest_face.json",
}
for path, expected_name in expected_output_names.items():
    if Path(path).name != expected_name:
        raise RuntimeError(f"Downstream filename changed: {path}")

for frame_name, frame in [
    ("item_docs", item_docs),
    ("item_facets", item_facets),
    ("graph_edges", graph_edges),
    ("item_review_timestamp_index", item_review_timestamp_index),
    ("item_daily_review_counts", item_daily_review_counts),
    ("entity_daily_review_counts", entity_daily_review_counts),
    ("item_temporal_summary", item_temporal_summary),
]:
    assert_no_leakage_columns(frame, frame_name)

for frame_name, frame in [
    ("item_review_timestamp_index", item_review_timestamp_index),
    ("item_daily_review_counts", item_daily_review_counts),
    ("entity_daily_review_counts", entity_daily_review_counts),
    ("item_temporal_summary", item_temporal_summary),
]:
    assert_no_raw_review_text_columns(frame, frame_name)

manifest = {
    "schema_path": str(SCHEMA_PATH),
    "item_docs_path": str(ITEM_DOCS_PATH),
    "items_facets_path": str(ITEMS_FACETS_PATH),
    "graph_edges_path": str(GRAPH_EDGES_PATH),
    "facet_vocab_path": str(FACET_VOCAB_PATH),
    "core_facet_filter_audit_path": str(CORE_FACET_FILTER_AUDIT_PATH),
    "brand_graph_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "global_review_graph_mask": GLOBAL_REVIEW_GRAPH_MASK_DESCRIPTION,
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_quarantined": False,
    "review_reputation_graph_enabled": True,
    "brand_in_functional_graph": False,
    "brand_graph_enabled": True,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
    "raw_review_text_exported": False,
    "dense_source": "dense_text",
    "sparse_source": "sparse_text",
    "item_facet_export_option": PRODUCTION_FACET_EXPORT_OPTION,
    "notebook_07_required_core_mask": CORE_GRAPH_MASK_DESCRIPTION,
    "notebook_07_required_brand_mask": BRAND_GRAPH_MASK_DESCRIPTION,
    "production_text_fields": ["dense_text", "sparse_text"],
    "production_aliases": {
        "dense_text_v2": "dense_text",
        "bm25_text": "sparse_text",
        "bm25_text_v2": "sparse_text",
    },
    "row_counts": {
        "active_item_docs": int(len(item_docs)),
        "all_item_facet_rows": int(len(item_facets)),
        "metadata_functional_facet_rows": int(len(core_item_facets)),
        "review_reputation_facet_rows": int(len(review_item_facets)),
        "metadata_graph_edges": int(graph_edges["is_core_graph_facet"].sum()),
        "brand_graph_edges": int(graph_edges["is_brand_graph_facet"].sum()),
        "review_reputation_graph_edges": int(graph_edges["is_review_derived"].sum()),
        "global_review_graph_edges": int(len(graph_edges)),
    },
    "production_graph_validation": {
        "review_derived_rows_used": int(graph_edges["is_review_derived"].sum()),
        "row_brand_value_rows_used": int(graph_edges["is_row_brand_value"].sum()),
        "brand_rows_used": int(graph_edges["is_brand"].sum()),
        "generic_anchor_rows_used": int(graph_edges["is_generic_category_anchor"].sum()),
        "generic_utility_rows_used": int(graph_edges["is_generic_utility_token"].sum()),
        "context_utility_rows_used": int(graph_edges["is_context_dependent_utility_token"].sum()),
        "disallowed_source_rows_used": int(graph_edges["is_disallowed_nonfacet_source"].sum()),
    },
    "discontinued_removed_count": discontinued_removed_count,
    "raw_review_text_loaded_for_temporal_artifacts": False,
    "rating_used": False,
    "sentiment_used": False,
}
with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print("Output:", MANIFEST_PATH)
print("Validation: Global Review retrieval artifacts passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/retrieval_artifact_manifest_face.json
Validation: Global Review retrieval artifacts passed


In [15]:
# ==== Common Retrieval Artifact Contract Export ====
facet_role_coverage_df.to_csv(
    COMMON_FACET_ROLE_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)

self_check_df = pd.DataFrame([{
    "item_rows": int(len(item_docs)),
    "all_facet_rows": int(len(item_facets)),
    "core_facet_rows": int(len(core_item_facets)),
    "brand_facet_rows": int(len(brand_item_facets)),
    "review_facet_rows": int(len(review_item_facets)),
    "global_review_facet_rows": int(len(global_review_item_facets)),
    "core_dense_non_empty": int(item_docs["dense_text_core"].fillna("").str.strip().ne("").sum()),
    "core_sparse_non_empty": int(item_docs["sparse_text_core"].fillna("").str.strip().ne("").sum()),
    "dense_source_mismatch_rows": int((item_docs["dense_text"] != item_docs["canonical_text_dense"]).sum()),
    "sparse_source_mismatch_rows": int((item_docs["sparse_text"] != item_docs["canonical_text_sparse"]).sum()),
    "row_brand_value_rows": int(item_facets["is_row_brand_value"].sum()),
    "brand_in_functional_rows": int(
        (item_facets["is_brand"] & item_facets["is_product_functional_facet"]).sum()
    ),
    "review_rows_in_production_graph": int(graph_edges["is_review_derived"].sum()),
    "metadata_rows_in_production_graph": int(graph_edges["is_core_graph_facet"].sum()),
    "brand_rows_in_production_graph": int(graph_edges["is_brand"].sum()),
    "generic_rows_in_production_graph": int(
        (graph_edges["is_generic_category_anchor"] | graph_edges["is_generic_utility_token"]).sum()
    ),
    "status": "PASS",
}])
self_check_df.to_csv(
    COMMON_FRAMEWORK_FLAG_SELF_CHECK_PATH,
    index=False,
    encoding="utf-8-sig",
)

contract = dict(COMMON_RETRIEVAL_ARTIFACT_CONTRACT)
contract["output_paths"] = {
    "item_docs": str(ITEM_DOCS_PATH),
    "items_facets": str(ITEMS_FACETS_PATH),
    "facet_vocab": str(FACET_VOCAB_TABLE_PATH),
    "graph_edges": str(GRAPH_EDGES_PATH),
    "core_facet_filter_audit": str(CORE_FACET_FILTER_AUDIT_PATH),
}
contract["production_text_fields"] = ["dense_text", "sparse_text"]
contract["brand_graph_enabled"] = True
contract["brand_in_functional_graph"] = False
contract["brand_in_retrieval_text"] = True
contract["brand_in_profile_source_text"] = True
contract["brand_in_synthetic_query"] = False
contract["production_aliases"] = {
    "dense_text_v2": "dense_text",
    "bm25_text": "sparse_text",
    "bm25_text_v2": "sparse_text",
}
contract["available_evidence_fields"] = [
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_derived_signal_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "dense_text",
    "sparse_text",
]
contract["facet_flags"] = [
    "is_row_brand_value",
    "is_brand",
    "is_review_derived",
    "is_query_safe",
    "is_product_functional_facet",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_specific_facet_phrase",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]
with open(COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH, "w", encoding="utf-8") as file:
    json.dump(contract, file, ensure_ascii=False, indent=2)

print("Output:", COMMON_RETRIEVAL_ARTIFACT_CONTRACT_PATH)
print("Validation: common Global Review retrieval contract passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_retrieval_artifact_common_contract.json
Validation: common Global Review retrieval contract passed


In [16]:
# ==== Debug: Global Review Retrieval Artifact Contract Checks - Face ====
dense_mismatch_rows = int(
    (
        item_docs["dense_text"].fillna("").astype(str) != item_docs["canonical_text_dense"].fillna("").astype(str)
    ).sum()
)
sparse_mismatch_rows = int(
    (
        item_docs["sparse_text"].fillna("").astype(str) != item_docs["canonical_text_sparse"].fillna("").astype(str)
    ).sum()
)

metadata_functional_facet_rows = int(len(core_item_facets))
review_facet_rows = int(len(review_item_facets))
metadata_graph_edges_debug = int(
    graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum()
)
review_graph_edges_debug = int(
    graph_edges["is_review_derived"].fillna(False).astype(bool).sum()
)
brand_graph_rows = int(
    graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum()
)

debug_contract_df = pd.DataFrame([
    {
        "check": "dense_text == canonical_text_dense",
        "passed": dense_mismatch_rows == 0,
        "observed": dense_mismatch_rows,
        "expected": 0,
    },
    {
        "check": "sparse_text == canonical_text_sparse",
        "passed": sparse_mismatch_rows == 0,
        "observed": sparse_mismatch_rows,
        "expected": 0,
    },
    {
        "check": "metadata functional facet rows > 0",
        "passed": metadata_functional_facet_rows > 0,
        "observed": metadata_functional_facet_rows,
        "expected": "> 0",
    },
    {
        "check": "historical review facet rows > 0",
        "passed": review_facet_rows > 0,
        "observed": review_facet_rows,
        "expected": "> 0",
    },
    {
        "check": "metadata graph edges > 0",
        "passed": metadata_graph_edges_debug > 0,
        "observed": metadata_graph_edges_debug,
        "expected": "> 0",
    },
    {
        "check": "review-derived graph edges > 0",
        "passed": review_graph_edges_debug > 0,
        "observed": review_graph_edges_debug,
        "expected": "> 0",
    },
    {
        "check": "brand graph rows > 0",
        "passed": brand_graph_rows > 0,
        "observed": brand_graph_rows,
        "expected": "> 0",
    },
])

display(debug_contract_df)

failed_debug_checks = debug_contract_df.loc[
    ~debug_contract_df["passed"],
    "check",
].tolist()
if failed_debug_checks:
    raise RuntimeError(
        f"Global Review retrieval artifact debug checks failed: {failed_debug_checks}"
    )

print("Face Global Review retrieval artifact debug checks passed.")


,check,passed,observed,expected
0,dense_text == canonical_text_dense,True,0,0
1,sparse_text == canonical_text_sparse,True,0,0
2,metadata functional facet rows > 0,True,618684,> 0
3,historical review facet rows > 0,True,284282,> 0
4,metadata graph edges > 0,True,618684,> 0
5,review-derived graph edges > 0,True,284282,> 0
6,brand graph rows > 0,True,74751,> 0


Face Global Review retrieval artifact debug checks passed.
